# RCA Cause-Tag Behaviour + Control-Action Details — one sheet per alarm tag

For every configured alarm tag, enrich its `DATA/RCA_<tag>_FI_7to9_top5.csv` (each row = one alarm
episode with its top-5 RCA cause tags) with, per alarm:

1. **Cause-tag pre-alarm behaviour** — for each `Cause 1–5` tag, whether its PV was *increasing*,
   *decreasing* or *stagnant* in the **90 min before the alarm**, written into the cause cell
   (e.g. `03PIC_1013.PV [increasing]`).
2. **Control-action details** — operator OP/SP/MODE moves in the alarm window, KG-filtered to tags
   knowledge-graph-relevant to that alarm tag:
   - `KG_operated_tags` / `eliminated_tags`
   - `action_details` — per KG-kept tag, chronological, direction ↑/↓ + net step (OP %, SP EU)
   - per tag-type **`… dir`** + **`… avg|step|`** columns (direction + average step magnitude)

**The individual RCA CSVs are read-only — nothing is written back to them.** Instead every tag becomes
one formatted sheet (green = increased, red = decreased, grey = no change; tag-type headers rotated;
`Episode number` = cluster_id frozen as the first column) in a single combined workbook:

`DATA/RCA_cause_behavior_control_actions_ALL_TAGS.xlsx`

## Tags processed (add more in the `TAG_CONFIGS` list)
| Tag | RCA CSV | Control-actions workbook | Loop PV parquet |
|-----|---------|--------------------------|-----------------|
| 03LIC_1071 (PVLO) | `RCA_1071_FI_7to9_top5.csv` | `…/03LIC_1071_PVLO_episodes_12JUN2026_1219/…` | `03LIC_1071_JAN_2026.parquet` |
| 03PIC_1104 (PVHI) | `RCA_1104_FI_7to9_top5.csv` | `…/03PIC_1104_PVHI_episodes_06JUL2026_1615/…` | `03PIC_1104_JAN_2026.parquet` |
| 03TIC_1023 (PVLO) | `RCA_1023_FI_7to9_top5.csv` | `…/03TIC_1023_episodes_03JUN2026_0915/…` | `03TIC_1023_JAN_2026.parquet` |
| 03LIC_1016 (PVLO) | `RCA_1016_FI_7to9_top5.csv` | `…/03LIC_1016_episodes_02JUN2026_1607/…` | `03LIC_1016_JAN_2026.parquet` |

## Shared references
| Source | Path | Role |
|--------|------|------|
| KG relevance | `DATA/Full_DB_Merged_Tag_Instrument_Sequences_V17Input 1.xlsx` | `DCS_Path` → per-tag KG tag set |
| PV history (gap-fill) | `DATA/new_rca_pv_op_data/merged_all_historian_tags.parquet` | fills any cause tag missing from a loop export (covers all cause tags) |
| Operating limits | `DATA/operating_limits.csv` | per-tag dead-band for the increasing/decreasing/stagnant call |

**Control-action window** = cluster_start − 30 min → cluster_end + 30 min. Every RCA alarm timestamp
is a `cluster_start_time`, so each alarm maps **exactly** to its cluster (verified: 100 % for all tags).
Every tag is processed with the **identical** engine — only the three inputs per tag differ.


In [11]:
# ═══ C1 · Config: tags to process, shared references, constants ════════════════
import os, shutil
import numpy as np
import pandas as pd
import pyarrow.parquet as _pq
from collections import Counter
from scipy.stats import theilslopes, norm
from IPython.display import display

DATA = "/home/h604827/ControlActions/DATA"
RES  = "/home/h604827/ControlActions/RESULTS"

# ── One entry per alarm tag. Append here as new RCA CSVs arrive. ────────────────
#   tag          : short id            full         : full PV-less tag (sheet name)
#   input_csv    : raw RCA file (Alarm_Timestamp + Cause 1-5) — READ-ONLY, never modified
#   ca_xlsx      : per-tag clustered-alarms + control_actions workbook
#   loop_parquet : per-loop PV/OP export (primary PV source; merged parquet gap-fills)
TAG_CONFIGS = [
    dict(tag="1071", full="03LIC_1071",
         input_csv=f"{DATA}/RCA_1071_FI_7to9_top5.csv",
         ca_xlsx=f"{RES}/03LIC_1071_PVLO_episodes_12JUN2026_1219/03LIC_1071_pvlo_alarms_clustered_with_control_actions.xlsx",
         loop_parquet=f"{DATA}/PV-OP_data/03LIC_1071_JAN_2026.parquet"),
    dict(tag="1104", full="03PIC_1104",
         input_csv=f"{DATA}/RCA_1104_FI_7to9_top5.csv",
         ca_xlsx=f"{RES}/03PIC_1104_PVHI_episodes_06JUL2026_1615/03PIC_1104_pvhi_alarms_clustered_with_control_actions.xlsx",
         loop_parquet=f"{DATA}/PV-OP_data/03PIC_1104_JAN_2026.parquet"),
    dict(tag="1023", full="03TIC_1023",
         input_csv=f"{DATA}/RCA_1023_FI_7to9_top5.csv",
         ca_xlsx=f"{RES}/03TIC_1023_episodes_03JUN2026_0915/03TIC_1023_pvlo_alarms_clustered_with_control_actions.xlsx",
         loop_parquet=f"{DATA}/PV-OP_data/03TIC_1023_JAN_2026.parquet"),
    dict(tag="1016", full="03LIC_1016",
         input_csv=f"{DATA}/RCA_1016_FI_7to9_top5.csv",
         ca_xlsx=f"{RES}/03LIC_1016_episodes_02JUN2026_1607/03LIC_1016_pvlo_alarms_clustered_with_control_actions.xlsx",
         loop_parquet=f"{DATA}/PV-OP_data/03LIC_1016_JAN_2026.parquet"),
    dict(tag="1009", full="03TIC_1009",
         input_csv=f"{DATA}/RCA_1009_FI_7to9_top5.csv",
         ca_xlsx=f"{RES}/03TIC_1009_PVLO_episodes_14JUN2026_1140/03TIC_1009_pvlo_alarms_clustered_with_control_actions.xlsx",
         loop_parquet=f"{DATA}/PV-OP_data/03TIC_1009_JAN_2026.parquet"),

]

# ── Shared references (identical for every tag) ────────────────────────────────
KG_XLSX        = f"{DATA}/Full_DB_Merged_Tag_Instrument_Sequences_V17Input 1.xlsx"
MERGED_PARQUET = f"{DATA}/new_rca_pv_op_data/merged_all_historian_tags.parquet"
OPLIM_CSV      = f"{DATA}/operating_limits.csv"

# ── Single combined output: one sheet per tag (individual CSVs are NOT edited) ──
OUTPUT_XLSX = f"{DATA}/RCA_cause_behavior_control_actions_ALL_TAGS.xlsx"

# ── Toggles / constants (identical treatment for every tag) ────────────────────
INCLUDE_DIRMAG_MATRIX = True         # per (tag,type) 'dir' + 'avg|step|' columns
WINDOW        = pd.Timedelta(minutes=30)   # control-action window pad each side of the alarm
PRE_MIN       = 90                          # minutes of pre-alarm PV context (cause behaviour)
WINDOWS       = [15, 30, 60, 90]            # multi-scale trend windows (min, ending at the alarm)
SMOOTH_MIN    = 5                           # rolling-median smoothing (min)
DEADBAND_FRAC = 0.05                        # dead-band = 5% of operating band ...
NOISE_K       = 2.0                         # ... or k x in-window noise, whichever is larger
STRONG_MULT   = 2.0                         # strong-ramp threshold (x dead-band)
CAUSE_COLS    = ["Cause 1", "Cause 2", "Cause 3", "Cause 4", "Cause 5"]
NO_ACTIONS_MSG = "No control actions in window"

# ── Shared lookups loaded once ─────────────────────────────────────────────────
_ol = pd.read_csv(OPLIM_CSV).set_index("TAG_NAME")
_kg_paths = pd.read_excel(KG_XLSX, sheet_name=0, usecols=["DCS_Path"])["DCS_Path"].dropna().astype(str)
_kg_token_sets = _kg_paths.apply(lambda p: {t.strip() for t in p.split(",") if t.strip()})

def load_pristine_alarms(input_csv):
    """Raw alarm table (Alarm_Timestamp + Cause 1-5) WITHOUT modifying input_csv.
    If a prior run enriched the CSV in place, read its one-time pristine backup instead."""
    backup = input_csv.replace(".csv", "_ORIGINAL_backup.csv")
    cur = pd.read_csv(input_csv)
    enriched = ("KG_operated_tags" in cur.columns) or ("Episode number" in cur.columns)
    if enriched:
        return pd.read_csv(backup) if os.path.exists(backup) else cur
    if not os.path.exists(backup):
        shutil.copy2(input_csv, backup)          # keep a pristine copy; original left untouched
    return cur

print(f"Tags to process : {[c['full'] for c in TAG_CONFIGS]}")
print(f"Operating limits: {_ol.shape[0]} tags   KG DCS_Path rows: {len(_kg_paths):,}")
print(f"Combined output : {OUTPUT_XLSX}")


Tags to process : ['03LIC_1071', '03PIC_1104', '03TIC_1023', '03LIC_1016', '03TIC_1009']
Operating limits: 40 tags   KG DCS_Path rows: 31,658
Combined output : /home/h604827/ControlActions/DATA/RCA_cause_behavior_control_actions_ALL_TAGS.xlsx


In [12]:
# ═══ C2 · Control-action rendering helpers (identical to the 1071 build) ═══════
_ARROW = {"increase": "↑", "decrease": "↓"}
_UNIT  = {"OP": "%", "SP": ""}
ACT_TYPES = ["OP", "SP"]

def _is_num(x):
    try:
        float(x); return True
    except (ValueError, TypeError):
        return False

def _fmt_val(x):
    return f"{round(float(x), 2):g}"

def _to_float(x):
    try:
        return float(x)
    except (ValueError, TypeError):
        return np.nan

def _action_detail(win):
    """One line per tag (ordered by first action); consecutive same type+direction numeric moves
    collapse to start->end (net, #moves). OP in %, SP in EU; OP/SP/MODE only."""
    df = win[win["Description"].isin(["OP", "SP", "MODE"])].sort_values("VT_Start")
    if df.empty:
        return ""
    first_time = df.groupby("Source")["VT_Start"].min().sort_values()
    lines = []
    for seq, (tag, t0) in enumerate(first_time.items(), 1):
        rows = df[df["Source"] == tag].sort_values("VT_Start").to_dict("records")
        segs = []; i = 0
        while i < len(rows):
            r = rows[i]; desc = r["Description"]; d = r["action_direction"]
            if desc == "MODE":
                segs.append(f"MODE {r['PrevValue']}→{r['Value']}"); i += 1
            elif _is_num(r["PrevValue"]) and _is_num(r["Value"]):
                j = i; last = r
                while (j + 1 < len(rows) and rows[j+1]["Description"] == desc
                       and rows[j+1]["action_direction"] == d
                       and _is_num(rows[j+1]["PrevValue"]) and _is_num(rows[j+1]["Value"])):
                    j += 1; last = rows[j]
                start = float(r["PrevValue"]); end = float(last["Value"]); n = j - i + 1
                net = round(end - start, 2); unit = _UNIT.get(desc, ""); ar = _ARROW.get(d, "")
                mv = f"{n} move" + ("s" if n > 1 else "")
                segs.append(f"{desc}{(' ' + ar) if ar else ''} {_fmt_val(start)}→{_fmt_val(end)} ({net:+g}{unit}, {mv})")
                i = j + 1
            else:
                j = i; last = r
                while (j + 1 < len(rows) and rows[j+1]["Description"] == desc
                       and not (_is_num(rows[j+1]["PrevValue"]) and _is_num(rows[j+1]["Value"]))):
                    j += 1; last = rows[j]
                n = j - i + 1
                segs.append(f"{desc} {r['PrevValue']}→{last['Value']}" + (f" ({n}x)" if n > 1 else ""))
                i = j + 1
        lines.append(f"{seq}. [{t0.strftime('%Y-%m-%d %H:%M')}] {tag} — " + ", then ".join(segs))
    return "\n".join(lines)

def _tagtype_summary(win, kg_set):
    """{(source, type): (direction, avg_abs_step, n_moves)} for KG-kept OP/SP numeric moves.
    Majority direction by #moves; net-change sign breaks ties."""
    df = win[win["Source"].isin(kg_set) & win["Description"].isin(ACT_TYPES)].copy()
    df["pv"] = df["PrevValue"].map(_to_float)
    df["vv"] = df["Value"].map(_to_float)
    df = df[df["pv"].notna() & df["vv"].notna()].sort_values("VT_Start")
    out = {}
    for (src, typ), grp in df.groupby(["Source", "Description"]):
        deltas = (grp["vv"] - grp["pv"]).to_numpy()
        inc, dec = int((deltas > 0).sum()), int((deltas < 0).sum())
        net = float(grp["vv"].iloc[-1] - grp["pv"].iloc[0])
        if inc > dec:
            direction = "increased"
        elif dec > inc:
            direction = "decreased"
        else:
            direction = ("increased" if net > 1e-9 else "decreased" if net < -1e-9 else "no change")
        out[(src, typ)] = (direction, round(float(np.abs(deltas).mean()), 2), len(deltas))
    return out

print("Control-action renderers ready: _action_detail, _tagtype_summary")


Control-action renderers ready: _action_detail, _tagtype_summary


In [13]:
# ═══ C3 · Trend engine: denoise -> multi-scale slope -> dead-band ══════════════
def _theil_slope(sub):
    """Theil-Sen slope (EU per minute) of a time-indexed Series; NaN if too few pts."""
    sub = sub.dropna()
    if len(sub) < 5:
        return np.nan
    x = (sub.index - sub.index[0]).total_seconds().to_numpy() / 60.0
    return theilslopes(sub.to_numpy(), x)[0]

def _mann_kendall(y):
    """Non-parametric monotonic-trend test. Returns (label, p_value)."""
    y = np.asarray(y, dtype=float)
    y = y[~np.isnan(y)]
    n = len(y)
    if n < 6:
        return ('insufficient', np.nan)
    s_stat = 0.0
    for i in range(n - 1):
        s_stat += np.sign(y[i + 1:] - y[i]).sum()
    var = n * (n - 1) * (2 * n + 5) / 18.0
    z = (s_stat - np.sign(s_stat)) / np.sqrt(var)
    p = 2 * (1 - norm.cdf(abs(z)))
    if p < 0.05 and z > 0:
        return ('increasing', float(p))
    if p < 0.05 and z < 0:
        return ('decreasing', float(p))
    return ('no-trend', float(p))

def _robust_sigma(resid):
    resid = resid[~np.isnan(resid)]
    if len(resid) < 3:
        return np.nan
    return 1.4826 * np.median(np.abs(resid - np.median(resid)))

def characterize_trend(pv_df, pv_col, cstart):
    """Characterise a cause tag's PV over the 90 min before the alarm (cstart), using pv_df."""
    anchor = pd.Timestamp(cstart).floor('min')
    grid = pd.date_range(anchor - pd.Timedelta(minutes=PRE_MIN), anchor, freq='1min')
    s = pv_df[pv_col].reindex(grid, method='nearest', tolerance=pd.Timedelta('30s'))
    coverage = float(s.notna().mean())

    out = dict(pv_col=pv_col, coverage=coverage, direction='no_data',
               trend_character='no_data', data_quality='insufficient_data',
               onset_time=pd.NaT)
    if s.notna().sum() < 10:
        return out

    # denoise (robust) + noise level from the residual
    s_filled = s.interpolate('linear', limit=SMOOTH_MIN, limit_direction='both')
    s_s = s_filled.rolling(SMOOTH_MIN, center=True, min_periods=2).median()
    sigma = _robust_sigma((s - s_s).to_numpy())

    # per-tag operating band + dead-band
    if pv_col in _ol.index:
        lo, hi = float(_ol.at[pv_col, 'LOWER_LIMIT']), float(_ol.at[pv_col, 'UPPER_LIMIT'])
        op_band = hi - lo if hi > lo else np.nan
    else:
        lo = hi = op_band = np.nan
    parts = []
    if np.isfinite(op_band): parts.append(DEADBAND_FRAC * op_band)
    if np.isfinite(sigma):   parts.append(NOISE_K * sigma)
    deadband = max(parts) if parts else max(NOISE_K * float(np.nanstd(s.to_numpy())), 1e-9)

    # multi-window robust slopes + net change (slope x window)
    slopes = {w: _theil_slope(s_s.loc[anchor - pd.Timedelta(minutes=w): anchor]) for w in WINDOWS}
    nets = {w: (slopes[w] * w if np.isfinite(slopes[w]) else np.nan) for w in WINDOWS}
    net90 = nets[90]
    passes = {w: (np.isfinite(nets[w]) and abs(nets[w]) > deadband) for w in WINDOWS}
    signs = {w: (np.sign(nets[w]) if np.isfinite(nets[w]) else 0) for w in WINDOWS}

    # six 15-min segment slopes -> count direction flips (oscillation)
    seg_signs = []
    for k in range(6):
        seg = s_s.loc[anchor - pd.Timedelta(minutes=15 * (k + 1)): anchor - pd.Timedelta(minutes=15 * k)]
        ms = _theil_slope(seg)
        if np.isfinite(ms) and abs(ms * 15) > deadband:
            seg_signs.append(np.sign(ms))
    n_sign_changes = sum(1 for a, b in zip(seg_signs, seg_signs[1:]) if a != b)

    mk_label, mk_p = _mann_kendall(s_s.to_numpy())

    # classify character + direction (priority order)
    active = [signs[w] for w in WINDOWS if passes[w]]
    # a *coherent recent move* = short/medium windows that clear the dead-band all agree in sign while
    # the full 90-min slope stays flat -> a genuine late move into the alarm; label BEFORE 'oscillating'.
    recent_pass_signs = {signs[w] for w in (15, 30, 60) if passes[w]}
    coherent_recent = (not passes[90] and any(passes[w] for w in (15, 30))
                       and len(recent_pass_signs) == 1)
    if not any(passes.values()):
        character, direction = 'flat', 'stagnant'
    elif coherent_recent:
        wp = min(w for w in (15, 30, 60) if passes[w])
        character = 'late_move'
        direction = 'increasing' if nets[wp] > 0 else 'decreasing'
    elif n_sign_changes >= 3 and not passes[90]:
        character, direction = 'oscillating', 'stagnant'
    elif passes[90] and len(set(active)) > 1:
        character = 'reversing'
        direction = 'increasing' if net90 > 0 else 'decreasing'
    elif not passes[90] and any(passes[w] for w in (15, 30)):
        wp = min(w for w in WINDOWS if passes[w])
        character = 'late_move'
        direction = 'increasing' if nets[wp] > 0 else 'decreasing'
    elif passes[90] and abs(net90) >= STRONG_MULT * deadband and mk_label in ('increasing', 'decreasing'):
        character = 'strong_ramp'
        direction = 'increasing' if net90 > 0 else 'decreasing'
    else:
        character = 'weak_drift'
        direction = 'increasing' if net90 > 0 else 'decreasing'

    # onset = last reversal before the final move (peak before a fall / trough before a rise)
    onset_time, from_onset_min, from_onset_eu = pd.NaT, np.nan, np.nan
    vser = s_s.dropna()
    v_alarm = float(vser.iloc[-1]) if len(vser) else np.nan
    v_start = float(vser.iloc[0]) if len(vser) else np.nan
    if direction in ('increasing', 'decreasing') and len(vser):
        onset_time = vser.idxmin() if direction == 'increasing' else vser.idxmax()
        from_onset_min = (anchor - onset_time).total_seconds() / 60.0
        from_onset_eu = v_alarm - float(vser.loc[onset_time])

    pos = ('below_low' if (np.isfinite(lo) and v_alarm < lo)
           else 'above_high' if (np.isfinite(hi) and v_alarm > hi)
           else 'within' if np.isfinite(lo) else 'unknown')

    out.update(
        data_quality=('ok' if coverage >= 0.5 else 'sparse'),
        direction=direction, trend_character=character,
        value_at_alarm=v_alarm, value_90min_ago=v_start,
        net90_eu=net90, pct_of_op_band=(100 * net90 / op_band if np.isfinite(op_band) else np.nan),
        slope15=slopes[15], slope30=slopes[30], slope60=slopes[60], slope90=slopes[90],
        net15=nets[15], net30=nets[30], net60=nets[60],
        mk_trend=mk_label, mk_p=mk_p,
        onset_time=onset_time, from_onset_min=from_onset_min, from_onset_eu=from_onset_eu,
        sigma_noise=sigma, op_band=op_band, deadband_eu=deadband,
        at_alarm_vs_limits=pos, n_sign_changes=n_sign_changes,
    )
    return out

def _behavior_word(r):
    """Short label written next to the tag inside the cause cell."""
    d = r["direction"]
    if d == "no_data":
        return "no data"
    if d == "stagnant":
        return "stagnant (oscillating)" if r["trend_character"] == "oscillating" else "stagnant"
    qual = {"late_move": " (late)", "reversing": " (reversing)", "weak_drift": " (weak)"}.get(
        r["trend_character"], "")
    return f"{d}{qual}"

print("Trend engine ready: characterize_trend(pv_df, pv_col, cstart), _behavior_word(r)")


Trend engine ready: characterize_trend(pv_df, pv_col, cstart), _behavior_word(r)


In [14]:
# ═══ C4 · Per-tag pipeline: KG set -> clusters -> control actions -> PV -> trends ═
def kg_tags_for(full):
    """Union of all tags on every DCS_Path that contains the target tag."""
    tags = set()
    for ts in _kg_token_sets[_kg_token_sets.apply(lambda x: full in x)]:
        tags |= ts
    return tags

def _clusters_and_actions(ca_xlsx):
    """cluster bounds (CB) + de-duplicated control actions (CA_DEDUP) from one workbook."""
    ac = pd.read_excel(ca_xlsx, sheet_name="alarm_clusters")
    ac["cluster_start_time"] = pd.to_datetime(ac["cluster_start_time"])
    ac["cluster_end_time"]   = pd.to_datetime(ac["cluster_end_time"])
    CB = (ac.groupby("cluster_id")
            .agg(cstart=("cluster_start_time", "min"), cend=("cluster_end_time", "max"))
            .sort_values("cstart"))
    CA = pd.read_excel(ca_xlsx, sheet_name="control_actions",
                       usecols=["cluster_id", "Source", "Description", "action_direction",
                                "PrevValue", "Value", "VT_Start"])
    CA["VT_Start"] = pd.to_datetime(CA["VT_Start"])
    CA = CA[CA["Source"].notna()].copy()
    CA_DEDUP = CA.drop_duplicates(subset=["Source", "Description", "VT_Start", "PrevValue", "Value"])
    return CB, CA_DEDUP

def _load_pv(needed_pv, loop_parquet):
    """PV history for needed cause tags: loop export primary, merged historian gap-fill."""
    def _read_avail(path, cols):
        names = set(_pq.read_schema(path).names)
        have = [c for c in cols if c in names]
        if not have:
            return None
        df = pd.read_parquet(path, columns=["TimeStamp", *have])
        df = df.dropna(subset=["TimeStamp"]).set_index("TimeStamp").sort_index()
        return df[~df.index.duplicated(keep="last")]
    pv = _read_avail(loop_parquet, needed_pv) if (loop_parquet and os.path.exists(loop_parquet)) else None
    if pv is None:
        pv = pd.DataFrame(index=pd.DatetimeIndex([], name="TimeStamp"))
    missing = [c for c in needed_pv if c not in pv.columns]
    if missing:
        m = _read_avail(MERGED_PARQUET, missing)
        if m is not None:
            pv = pv.join(m, how="outer")
    for c in needed_pv:
        if c not in pv.columns:
            pv[c] = np.nan
    return pv.astype("float64").sort_index(), missing

def process_tag(cfg):
    """Run the full 1071-style enrichment for one tag; return the assembled sheet + diagnostics."""
    full = cfg["full"]
    alarms = load_pristine_alarms(cfg["input_csv"])
    alarms["Alarm_Timestamp"] = pd.to_datetime(alarms["Alarm_Timestamp"])
    alarms = alarms.sort_values("Alarm_Timestamp").reset_index(drop=True)
    KG_TAGS = kg_tags_for(full)

    # clusters + EXACT minute alarm->cluster mapping (nearest kept only as a safety net)
    CB, CA_DEDUP = _clusters_and_actions(cfg["ca_xlsx"])
    CB["cstart_min"] = CB["cstart"].dt.floor("min")
    cstart_to_cid = {t: cid for cid, t in CB["cstart_min"].items()}
    cs = CB["cstart"].to_numpy().astype("datetime64[ns]")
    cid_arr = CB.index.to_numpy()
    def _win(alarm_ts):
        tmin = pd.Timestamp(alarm_ts).floor("min")
        cid = cstart_to_cid.get(tmin)
        if cid is not None:
            return (CB.at[cid, "cstart"] - WINDOW, CB.at[cid, "cend"] + WINDOW, int(cid), "cluster")
        i = int(np.abs(cs - np.datetime64(tmin, "ns")).argmin())
        return (CB.iloc[i]["cstart"] - WINDOW, CB.iloc[i]["cend"] + WINDOW, int(cid_arr[i]), "nearest")
    _w = alarms["Alarm_Timestamp"].apply(_win)
    alarms["win_start"]  = _w.apply(lambda x: x[0])
    alarms["win_end"]    = _w.apply(lambda x: x[1])
    alarms["cluster_id"] = _w.apply(lambda x: x[2])
    alarms["win_method"] = _w.apply(lambda x: x[3])

    # per-alarm control-action fields (absolute-time window on de-dup actions, KG-filtered)
    kept_list, elim_list, detail_list, ttsummary_list = [], [], [], []
    tagtype_freq = Counter()
    for _, row in alarms.iterrows():
        win = CA_DEDUP[(CA_DEDUP["VT_Start"] >= row["win_start"]) &
                       (CA_DEDUP["VT_Start"] <= row["win_end"])]
        operated = sorted(win["Source"].unique())
        if not operated:
            kept_list.append(NO_ACTIONS_MSG); elim_list.append(""); detail_list.append("")
            ttsummary_list.append({}); continue
        kept = [t for t in operated if t in KG_TAGS]
        elim = [t for t in operated if t not in KG_TAGS]
        kept_list.append(", ".join(kept)); elim_list.append(", ".join(elim))
        detail_list.append(_action_detail(win[win["Source"].isin(KG_TAGS)]))
        summ = _tagtype_summary(win, KG_TAGS); ttsummary_list.append(summ); tagtype_freq.update(summ.keys())
    alarms["KG_operated_tags"] = kept_list
    alarms["eliminated_tags"]  = elim_list
    alarms["action_details"]   = detail_list

    # PV history for the cause tags, then characterise each cause behaviour (per-tag cache)
    needed_pv = sorted({v.strip() for c in CAUSE_COLS for v in alarms[c].dropna().astype(str)
                        if v.strip().endswith(".PV")})
    pv_df, gapfilled = _load_pv(needed_pv, cfg["loop_parquet"])
    cache = {}
    def _char(pv_col, anchor):
        k = (pv_col, pd.Timestamp(anchor).floor("min"))
        if k not in cache:
            cache[k] = characterize_trend(pv_df, pv_col, anchor)
        return cache[k]

    enriched = alarms.copy()
    metric_rows = []
    for i, row in alarms.iterrows():
        anchor = row["Alarm_Timestamp"]
        for slot, col in enumerate(CAUSE_COLS, start=1):
            tag = row[col]
            if not isinstance(tag, str) or not tag.strip().endswith(".PV"):
                continue
            tag = tag.strip()
            res = _char(tag, anchor)
            metric_rows.append(dict(alarm_row=i, alarm_timestamp=anchor, cause_slot=slot,
                                    cause_col=col, **res))
            enriched.at[i, col] = f"{tag} [{_behavior_word(res)}]"
    metrics = pd.DataFrame(metric_rows)
    if len(metrics):
        metrics["behavior"] = metrics.apply(_behavior_word, axis=1)

    # per (tag,type) direction + avg|step| columns (most-operated first)
    dirmag_cols = []
    if INCLUDE_DIRMAG_MATRIX:
        order = [k for k, _ in sorted(tagtype_freq.items(), key=lambda kv: (-kv[1], kv[0]))]
        for (src, typ) in order:
            dcol, mcol = f"{src}.{typ} dir", f"{src}.{typ} avg|step|"
            enriched[dcol] = [s.get((src, typ), ("", np.nan, 0))[0] for s in ttsummary_list]
            enriched[mcol] = [s.get((src, typ), ("", np.nan, 0))[1] for s in ttsummary_list]
            dirmag_cols += [dcol, mcol]

    # assemble the sheet: Episode number (=cluster_id) first, then causes, meta, dir/mag matrix
    meta_cols = ["KG_operated_tags", "eliminated_tags", "action_details"]
    out = enriched[["Alarm_Timestamp"] + CAUSE_COLS + meta_cols + dirmag_cols].copy()
    out.insert(0, "Episode number", enriched["cluster_id"].astype("Int64").values)
    out["Alarm_Timestamp"] = pd.to_datetime(out["Alarm_Timestamp"]).dt.strftime("%Y-%m-%d %H:%M:%S")

    return dict(full=full, out=out, metrics=metrics, pv_df=pv_df,
                n_alarms=len(alarms), n_cluster=int((alarms["win_method"] == "cluster").sum()),
                n_with_actions=sum(1 for k in kept_list if k != NO_ACTIONS_MSG),
                n_dirmag=len(dirmag_cols) // 2, needed_pv=needed_pv, gapfilled=gapfilled)

print("process_tag(cfg) ready.")


process_tag(cfg) ready.


In [15]:
# ═══ C5 · Build the combined workbook — one coloured sheet per tag ═════════════
# Same formatting as the earlier single-tag deliverable, applied to every sheet:
#   direction cells green=increased / red=decreased / grey=no change; tag-type headers rotated
#   vertical; action_details wrapped; Episode number (=cluster_id) + top row frozen.
import openpyxl
from openpyxl.styles import PatternFill, Alignment, Font

DIR_FILL = {
    "increased": PatternFill("solid", fgColor="C6EFCE"),   # green
    "decreased": PatternFill("solid", fgColor="FFC7CE"),   # red
    "no change": PatternFill("solid", fgColor="D9D9D9"),   # grey
}
VERT = Alignment(text_rotation=90, vertical="bottom", horizontal="center")

def format_sheet(ws):
    """Colour/rotate/freeze one worksheet in place; returns (tag-type pairs, coloured cells)."""
    hdr = {ws.cell(row=1, column=c).value: c for c in range(1, ws.max_column + 1)}
    dirmag_pairs = []
    for h in hdr:
        if isinstance(h, str) and h.endswith(" dir"):
            mh = f"{h[:-4]} avg|step|"
            if mh in hdr:
                dirmag_pairs.append((h, mh))
    for dcol, mcol in dirmag_pairs:                       # rotate + bold tag-type headers
        for name in (dcol, mcol):
            cell = ws.cell(row=1, column=hdr[name]); cell.alignment = VERT; cell.font = Font(bold=True)
        ws.column_dimensions[ws.cell(row=1, column=hdr[dcol]).column_letter].width = 11
        ws.column_dimensions[ws.cell(row=1, column=hdr[mcol]).column_letter].width = 9
    for name, width in [("KG_operated_tags", 26), ("eliminated_tags", 22), ("action_details", 72)]:
        if name in hdr:
            ws.column_dimensions[ws.cell(row=1, column=hdr[name]).column_letter].width = width
    if "action_details" in hdr:
        for r in range(2, ws.max_row + 1):
            ws.cell(row=r, column=hdr["action_details"]).alignment = Alignment(wrap_text=True, vertical="top")
    n_filled = 0                                          # colour every operated tag-type cell
    for dcol, mcol in dirmag_pairs:
        dci, mci = hdr[dcol], hdr[mcol]
        for r in range(2, ws.max_row + 1):
            fill = DIR_FILL.get(ws.cell(row=r, column=dci).value)
            if fill:
                ws.cell(row=r, column=dci).fill = fill
                ws.cell(row=r, column=mci).fill = fill
                n_filled += 1
    ws.row_dimensions[1].height = 120
    if "Episode number" in hdr:
        ws.column_dimensions[ws.cell(row=1, column=hdr["Episode number"]).column_letter].width = 11
    ws.freeze_panes = "B2"
    return len(dirmag_pairs), n_filled

# process every configured tag, then write + format one sheet each
RESULTS_BY_TAG = {}
for cfg in TAG_CONFIGS:
    res = process_tag(cfg)
    RESULTS_BY_TAG[cfg["full"]] = res
    warn = "" if res["n_cluster"] == res["n_alarms"] else f"  ⚠ {res['n_alarms'] - res['n_cluster']} not exact-matched"
    print(f"{res['full']:12s}: {res['n_alarms']:>4} alarms | {res['n_with_actions']:>4} with actions | "
          f"{res['n_dirmag']:>2} tag-type pairs | {res['out'].shape[1]} cols{warn}")

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    for full, res in RESULTS_BY_TAG.items():
        res["out"].to_excel(writer, sheet_name=full[:31], index=False)

wb = openpyxl.load_workbook(OUTPUT_XLSX)
for full, res in RESULTS_BY_TAG.items():
    pairs, filled = format_sheet(wb[full[:31]])
    print(f"  sheet {full[:31]:12s}: {res['out'].shape[0]:>4} rows x {res['out'].shape[1]:>3} cols  "
          f"{pairs} pairs, {filled} coloured cells")
wb.save(OUTPUT_XLSX)
print(f"\nWrote combined workbook -> {OUTPUT_XLSX}   ({len(RESULTS_BY_TAG)} sheets)")


03LIC_1071  :  360 alarms |  304 with actions | 41 tag-type pairs | 92 cols
03PIC_1104  :    7 alarms |    7 with actions | 33 tag-type pairs | 76 cols
03TIC_1023  :  619 alarms |  400 with actions | 19 tag-type pairs | 48 cols
03LIC_1016  :  373 alarms |  208 with actions | 23 tag-type pairs | 56 cols
03TIC_1009  :   23 alarms |   14 with actions |  9 tag-type pairs | 28 cols
  sheet 03LIC_1071  :  360 rows x  92 cols  41 pairs, 763 coloured cells
  sheet 03PIC_1104  :    7 rows x  76 cols  33 pairs, 54 coloured cells
  sheet 03TIC_1023  :  619 rows x  48 cols  19 pairs, 216 coloured cells
  sheet 03LIC_1016  :  373 rows x  56 cols  23 pairs, 363 coloured cells
  sheet 03TIC_1009  :   23 rows x  28 cols  9 pairs, 19 coloured cells

Wrote combined workbook -> /home/h604827/ControlActions/DATA/RCA_cause_behavior_control_actions_ALL_TAGS.xlsx   (5 sheets)


In [16]:
# ═══ C6 · Optional visual validation — 90-min pre-alarm window for any tag/row ═══
import plotly.graph_objects as go

def _vmark(fig, xts, color, text, dash="dot"):
    fig.add_shape(type="line", x0=xts, x1=xts, y0=0, y1=1, yref="paper",
                  line=dict(color=color, dash=dash, width=1.5))
    fig.add_annotation(x=xts, y=1.02, yref="paper", yanchor="bottom", text=text,
                       showarrow=False, font=dict(color=color, size=11))

def plot_cause_trend(full, i):
    """Plot raw + smoothed PV for row i of tag `full`'s metrics, with Theil-Sen line,
    dead-band, onset and operating limits."""
    res = RESULTS_BY_TAG[full]; metrics = res["metrics"]; pv_df = res["pv_df"]
    r = metrics.iloc[i]
    anchor = pd.Timestamp(r["alarm_timestamp"]).floor("min")
    grid = pd.date_range(anchor - pd.Timedelta(minutes=PRE_MIN), anchor, freq="1min")
    s = pv_df[r["pv_col"]].reindex(grid, method="nearest", tolerance=pd.Timedelta("30s"))
    s_s = (s.interpolate("linear", limit=SMOOTH_MIN, limit_direction="both")
             .rolling(SMOOTH_MIN, center=True, min_periods=2).median())
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=grid, y=s, mode="lines", name="raw PV",
                             line=dict(color="lightgray", width=1)))
    fig.add_trace(go.Scatter(x=grid, y=s_s, mode="lines", name="smoothed (5-min median)",
                             line=dict(color="royalblue", width=2)))
    base = s_s.dropna()
    if len(base) and np.isfinite(r.get("slope90", np.nan)):
        b0 = base.iloc[0]; xm = (grid - grid[0]).total_seconds() / 60.0
        fig.add_trace(go.Scatter(x=grid, y=b0 + r["slope90"] * xm, mode="lines",
                                 name="Theil-Sen 90-min", line=dict(color="firebrick", dash="dash")))
        if np.isfinite(r.get("deadband_eu", np.nan)):
            for sgn in (+1, -1):
                fig.add_trace(go.Scatter(x=grid, y=[b0 + sgn * r["deadband_eu"]] * len(grid),
                                         mode="lines", line=dict(color="gray", dash="dot", width=1),
                                         name="dead-band", showlegend=(sgn == 1)))
    if r["pv_col"] in _ol.index:
        for lim, nm in [(_ol.at[r["pv_col"], "LOWER_LIMIT"], "low lim"),
                        (_ol.at[r["pv_col"], "UPPER_LIMIT"], "high lim")]:
            fig.add_hline(y=float(lim), line=dict(color="orange", dash="dot", width=1),
                          annotation_text=nm)
    if not pd.isna(r.get("onset_time", pd.NaT)):
        _vmark(fig, pd.Timestamp(r["onset_time"]), "green", "onset")
    _vmark(fig, anchor, "black", "alarm", dash="solid")
    fig.update_layout(
        title=(f"{full} · row {r['alarm_row']} · {r['cause_col']} · {r['pv_col']} · "
               f"alarm {anchor:%Y-%m-%d %H:%M}<br><sub>{_behavior_word(r)} "
               f"[{r['trend_character']}] · Δ90={r.get('net90_eu', float('nan')):+.2f} · "
               f"cov {r['coverage']:.0%}</sub>"),
        height=460, template="plotly_white", xaxis_title="time",
        yaxis_title=r["pv_col"], legend=dict(orientation="h", y=-0.2))
    return fig

# one example per distinct trend character for the first tag (skip no_data)
_full0 = TAG_CONFIGS[0]["full"]
_m0 = RESULTS_BY_TAG[_full0]["metrics"]
_seen, _n = set(), 0
for _i in range(len(_m0)):
    ch = _m0.iloc[_i]["trend_character"]
    if ch in _seen or ch == "no_data":
        continue
    _seen.add(ch); _n += 1
    plot_cause_trend(_full0, _i).show()
    if _n >= 6:
        break
print(f"Plotted {_n} example(s) for {_full0}. Use plot_cause_trend(full, i) for any tag/row.")


Plotted 6 example(s) for 03LIC_1071. Use plot_cause_trend(full, i) for any tag/row.
